# 01 — Exploratory Data Analysis
**Project:** Credit Card Fraud Detection  
**Dataset:** Kaggle `creditcard.csv` — 284,807 transactions, 492 fraud cases (0.17%)  
**Topics covered:**
- Class distribution & imbalance
- Correlation heatmap (V1–V28, Amount, Time)
- Time-of-day fraud patterns
- Transaction amount distributions by class

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_theme(style='whitegrid', palette='muted')
DATA_PATH = os.path.join('..', 'data', 'creditcard.csv')

## 1. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Fraud cases: {df["Class"].sum():,} ({df["Class"].mean():.4%})')
df.head()

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

## 2. Class Distribution

In [ ]:
class_counts = df['Class'].value_counts().reset_index()
class_counts.columns = ['Class', 'Count']
class_counts['Label'] = class_counts['Class'].map({0: 'Legitimate', 1: 'Fraud'})
class_counts['Percent'] = (class_counts['Count'] / class_counts['Count'].sum() * 100).round(4)

fig = px.bar(
    class_counts,
    x='Label', y='Count',
    color='Label',
    text=class_counts['Percent'].astype(str) + '%',
    color_discrete_map={'Legitimate': '#4C72B0', 'Fraud': '#DD5B5B'},
    title='Class Distribution — Severe Imbalance (0.17% Fraud)',
    template='plotly_white'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, yaxis_title='Transaction Count')
fig.show()

In [ ]:
fig_pie = px.pie(
    class_counts,
    names='Label', values='Count',
    color='Label',
    color_discrete_map={'Legitimate': '#4C72B0', 'Fraud': '#DD5B5B'},
    title='Class Proportion',
    hole=0.4,
    template='plotly_white'
)
fig_pie.show()

## 3. Correlation Heatmap

In [ ]:
corr = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=False, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.3, ax=ax
)
ax.set_title('Feature Correlation Matrix (lower triangle)', fontsize=16, pad=16)
plt.tight_layout()
plt.show()

In [ ]:
# Top features correlated with Class
class_corr = corr['Class'].drop('Class').sort_values()

fig = px.bar(
    x=class_corr.values,
    y=class_corr.index,
    orientation='h',
    color=class_corr.values,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    title='Feature Correlation with Fraud Class',
    labels={'x': 'Pearson Correlation', 'y': 'Feature'},
    template='plotly_white'
)
fig.update_layout(coloraxis_showscale=False, height=700)
fig.show()

## 4. Time Patterns — Fraud by Hour of Day

In [ ]:
df['Hour'] = (df['Time'] // 3600) % 24

hourly = df.groupby(['Hour', 'Class']).size().reset_index(name='Count')
hourly['Label'] = hourly['Class'].map({0: 'Legitimate', 1: 'Fraud'})

# Fraud rate per hour
fraud_rate = df.groupby('Hour')['Class'].mean().reset_index()
fraud_rate.columns = ['Hour', 'FraudRate']
fraud_rate['FraudRate'] = fraud_rate['FraudRate'] * 100

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('Transaction Volume by Hour', 'Fraud Rate (%) by Hour'),
    vertical_spacing=0.12
)

for label, color in [('Legitimate', '#4C72B0'), ('Fraud', '#DD5B5B')]:
    subset = hourly[hourly['Label'] == label]
    fig.add_trace(
        go.Bar(x=subset['Hour'], y=subset['Count'], name=label, marker_color=color),
        row=1, col=1
    )

fig.add_trace(
    go.Scatter(
        x=fraud_rate['Hour'], y=fraud_rate['FraudRate'],
        mode='lines+markers', name='Fraud Rate %',
        line=dict(color='#DD5B5B', width=2)
    ),
    row=2, col=1
)

fig.update_layout(
    title='Fraud Patterns by Hour of Day',
    template='plotly_white', height=600,
    xaxis2_title='Hour of Day (0–23)',
    barmode='stack'
)
fig.show()

In [ ]:
# Seaborn KDE: Time distribution by class
fig, ax = plt.subplots(figsize=(14, 5))
for cls, label, color in [(0, 'Legitimate', '#4C72B0'), (1, 'Fraud', '#DD5B5B')]:
    sns.kdeplot(df[df['Class'] == cls]['Hour'], fill=True, label=label, color=color, alpha=0.5, ax=ax)
ax.set_title('Hour-of-Day Distribution by Class', fontsize=14)
ax.set_xlabel('Hour of Day')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Amount Patterns

In [ ]:
fig = px.box(
    df[df['Amount'] < 2000],  # clip extreme outliers for visibility
    x=df[df['Amount'] < 2000]['Class'].map({0: 'Legitimate', 1: 'Fraud'}),
    y='Amount',
    color=df[df['Amount'] < 2000]['Class'].map({0: 'Legitimate', 1: 'Fraud'}),
    color_discrete_map={'Legitimate': '#4C72B0', 'Fraud': '#DD5B5B'},
    title='Transaction Amount Distribution by Class (Amount < $2,000)',
    labels={'x': 'Class', 'y': 'Amount (USD)'},
    template='plotly_white'
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Log-scale histogram overlay
fig, ax = plt.subplots(figsize=(12, 5))
for cls, label, color in [(0, 'Legitimate', '#4C72B0'), (1, 'Fraud', '#DD5B5B')]:
    subset = df[df['Class'] == cls]['Amount'] + 1  # +1 to allow log of 0
    ax.hist(subset, bins=80, alpha=0.6, label=label, color=color, log=True)
ax.set_xscale('log')
ax.set_title('Transaction Amount (log scale) by Class', fontsize=14)
ax.set_xlabel('Amount (USD) — log scale')
ax.set_ylabel('Count (log scale)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Summary stats by class
df.groupby('Class')['Amount'].describe().style.background_gradient(cmap='Oranges')

## 6. Top PCA Feature Distributions (V1–V10 by Class)

In [ ]:
top_features = ['V1', 'V2', 'V3', 'V4', 'V7', 'V9', 'V10', 'V11', 'V12', 'V14']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    for cls, label, color in [(0, 'Legit', '#4C72B0'), (1, 'Fraud', '#DD5B5B')]:
        sns.kdeplot(
            df[df['Class'] == cls][feat],
            ax=axes[i], label=label, color=color, fill=True, alpha=0.4
        )
    axes[i].set_title(feat, fontsize=12)
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('')

plt.suptitle('PCA Feature Distributions by Class', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 7. Missing Values & Data Quality

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values — dataset is clean.')

print(f'\nDuplicate rows: {df.duplicated().sum()}')
print(f'Amount range: ${df["Amount"].min():.2f} – ${df["Amount"].max():,.2f}')

## Key EDA Takeaways

| Finding | Implication |
|---|---|
| 0.17% fraud rate | Accuracy is a misleading metric — use AUC-ROC and Recall |
| Fraud peaks at hours 0–6 | Hour feature will be predictive |
| Fraud amounts skew lower (median ~$22) | Amount is useful but not decisive |
| V4, V11, V14, V17 strongly correlated with Class | These will dominate feature importance |
| No missing values | No imputation needed |

**Next step:** `src/preprocess.py` → scale Amount/Hour, apply SMOTE, split → `src/train.py`